In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/UGC-5823_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/UGC-5823_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     377   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        58   (2094, 964)   float32   


In [3]:
image_cut = image[48:168, 0:1890]
image_error_cut = image_error[48:168, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.0188
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5, params6, params7):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) +
        sersic_1d(x_hr, params6) +
        sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/PSF/Real_seeing_UGC5823_1.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 120, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, params6, params7, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5, params6, params7)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, params6, params7, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    purple_area = single_integral(x, params6, a, b)
    pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area + purple_area + pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, params6, params7, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area, (purple_area*100)/sum_area, (pink_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.35766511043283805
0.38443580582250503
0.8255004985487983
0.8247773620202423
1.0426031946138323
1.0534094732416168
7.029582135497247
4.595279083940769
0.7462149959947887
0.7446396514975457
1.5393311119550133
1.5379658671239005
2.2912168610879213
2.2343745807839275
                 Peak 1        Peak 2        Peak 3     Peak 4         Peak 5  \
Blue       9.383054e+01  3.468986e-01  1.905239e-01   0.030083   8.179859e-02   
Green      1.147317e+00  6.916059e+01  1.343064e+01   0.890683   1.187954e+00   
Magenta    1.626916e-05  8.834409e+00  5.629411e+01   0.375183   1.057829e-03   
Orange     4.994032e+00  2.060538e+01  2.863011e+01  97.519784   2.539683e+01   
Chocolate  2.809274e-02  1.052721e+00  1.454614e+00   1.184261   7.323560e+01   
Purple     0.000000e+00  0.000000e+00  0.000000e+00   0.000000  3.033560e-113   
Deeppink   3.074953e-12  2.224484e-07  9.189056e-07   0.000006   9.676828e-02   

              Peak 6        Peak 7  
Blue        0.018385  4.186298e-03  
Green      

In [10]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.0147669827079067
1.0814782634662286
2.393771395091222
2.386662749920247
3.01125435093778
3.037209059532665
11.224814972232714
7.885743789943646
1.4629319936641954
1.4610361780472059
3.0966196620037563
3.0939665635835727
6.426782614099864
6.280968393364437
                 Peak 1        Peak 2        Peak 3     Peak 4        Peak 5  \
Blue       9.346862e+01  3.598273e-01  1.983503e-01   0.037819  8.366106e-02   
Green      1.216166e+00  6.693607e+01  1.443219e+01   1.132893  1.218101e+00   
Magenta    1.795331e-05  1.017175e+01  5.378395e+01   0.547371  1.184525e-03   
Orange     5.285318e+00  2.143800e+01  3.006766e+01  96.786677  2.607195e+01   
Chocolate  2.988271e-02  1.094348e+00  1.517846e+00   1.495233  7.251608e+01   
Purple     0.000000e+00  0.000000e+00  0.000000e+00   0.000000  1.249764e-31   
Deeppink   3.381203e-12  2.402835e-07  9.973244e-07   0.000009  1.090249e-01   

              Peak 6        Peak 7  
Blue        0.018316  4.481625e-03  
Green       0.212661  4.793

In [11]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

1.4918676137569011
1.564160409912725
3.7046557256093
3.6913476658316906
4.679730930073355
4.704495928056523
13.649524950801219
10.113595342968944
2.140710324942824
2.138933137760679
4.626552321587888
4.623491720131083
9.285232248653129
9.104707411885881
                 Peak 1        Peak 2     Peak 3     Peak 4        Peak 5  \
Blue       9.257175e+01  3.902388e-01   0.214027   0.046939  8.612452e-02   
Green      1.387804e+00  6.098065e+01  17.091624   1.434309  1.259323e+00   
Magenta    2.265601e-05  1.400128e+01  47.835144   0.862332  1.409440e-03   
Orange     6.006009e+00  2.343431e+01  33.211674  95.787346  2.700898e+01   
Chocolate  3.441261e-02  1.193515e+00   1.647530   1.869062  7.151325e+01   
Purple     0.000000e+00  0.000000e+00   0.000000   0.000000  3.232857e-09   
Deeppink   4.229998e-12  2.883165e-07   0.000001   0.000012  1.309173e-01   

              Peak 6        Peak 7  
Blue        0.018453  5.183070e-03  
Green       0.214705  5.550304e-02  
Magenta     0.0000

## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.13900954527003517
0.13956943712085418
0.38900734792986275
0.38911968075613335
0.43974545597560905
0.41797646688624024
1.1354751925081403
1.1624626032765701
0.24323899148366424
0.24329716061115614
0.37114643030156286
0.3931887068634229
0.554448341811004
0.5557016545297746
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.576014e+01  2.445160e+00  1.847004e+00  4.826537e-01   
Green      1.902856e-15  6.475823e+01  3.169081e+01  4.868628e-02   
Magenta    0.000000e+00  2.195625e-01  2.432062e+01  0.000000e+00   
Orange     3.239931e+00  1.747279e+01  2.512551e+01  8.827372e+01   
Chocolate  9.997413e-01  1.510092e+01  1.701145e+01  1.118856e+01   
Purple     1.898599e-04  3.337912e-03  4.603211e-03  6.383503e-03   
Deeppink   2.407057e-93  1.655471e-42  2.475259e-38  3.358800e-28   

                 Peak 5        Peak 6        Peak 7  
Blue       1.121227e+00  4.431608e-01  2.298271e-01  
Green      2.907535e-12  1.264540e-26  2.709483e-36  
Magenta    

In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)
df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.4046237893583228
0.4057463212593911
1.1539402374665053
1.1530358500466584
1.6386248233325302
1.5760689302691382
3.2023559106371877
3.261928939021506
0.7293762293231962
0.7295501357077947
1.0648768413940015
1.1196139237634357
1.5802662667126488
1.5828313795742015
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.562059e+01  2.475131e+00  1.935775e+00  5.137566e-01   
Green      6.261809e-15  6.359303e+01  2.913629e+01  6.399446e-02   
Magenta    0.000000e+00  8.310460e-01  1.940649e+01  0.000000e+00   
Orange     3.343109e+00  1.780770e+01  3.051483e+01  8.751315e+01   
Chocolate  1.036100e+00  1.528970e+01  1.900122e+01  1.190225e+01   
Purple     1.963230e-04  3.395258e-03  5.402024e-03  6.844405e-03   
Deeppink   4.989324e-92  1.305901e-41  2.733705e-36  1.870326e-27   

                 Peak 5        Peak 6        Peak 7  
Blue       1.122302e+00  4.635577e-01  2.419989e-01  
Green      6.928399e-12  6.057112e-26  1.898167e-35  
Magenta    0.000000e

In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


0.6273631013323194
0.6282698345199765
1.88083922656826
1.8714183210698336
2.382588562976834
2.3166029252483487
4.581069066765484
4.63642117506315
1.2142483981147296
1.2145381594302453
1.6253730878490562
1.6834602731718054
2.320765960313262
2.322792985319638
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       9.526516e+01  2.537107e+00  2.002474e+00  5.996423e-01   
Green      3.811063e-14  6.027329e+01  3.113292e+01  1.197899e-01   
Magenta    0.000000e+00  2.919207e+00  1.412974e+01  3.099518e-29   
Orange     3.604610e+00  1.858466e+01  3.307664e+01  8.540316e+01   
Chocolate  1.130020e+00  1.568221e+01  1.965253e+01  1.386926e+01   
Purple     2.128678e-04  3.524779e-03  5.689495e-03  8.141907e-03   
Deeppink   2.222127e-90  2.060662e-40  4.169584e-35  2.184280e-26   

                 Peak 5        Peak 6        Peak 7  
Blue       1.125054e+00  5.067198e-01  2.749093e-01  
Green      3.192509e-11  6.786856e-25  3.354400e-34  
Magenta    0.000000e+00  0.

## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.11343825143577287
0.11585204457934423
0.30007165431529353
0.30070140968354314
0.36806267438616347
0.41688128800204133
0.8180232177366858
0.8044922498118385
0.2949690717577977
0.29508288094465546
0.2864510226699546
0.2863826264508703
0.9146735661614334
0.8799844550004363
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       8.582394e+01  2.441261e+00  1.601856e+00  4.816747e-01   0.680945   
Green      9.973354e-03  4.587638e+01  9.748617e+00  2.669455e-01   0.007697   
Magenta    4.088367e-05  1.570633e+00  2.591946e+01  3.988552e-02   0.000321   
Orange     8.413400e+00  3.395220e+01  4.575774e+01  8.672753e+01  23.224538   
Chocolate  5.752648e+00  1.615953e+01  1.697233e+01  1.248396e+01  76.086405   
Purple     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   
Deeppink   3.417961e-24  5.447616e-14  1.095370e-12  2.845950e-10   0.000093   

              Peak 6     Peak 7  
Blue        0.385042   0.196328  
Green       0.0001

In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.3314055647884468
0.3373727905067142
0.8839241759094824
0.8856224770374439
1.0817332862596594
1.204046839763879
2.359150809330154
2.321283418496108
0.8807162653468897
0.8810414648534063
0.8609764276158887
0.8607876448990998
1.625235802917542
1.5789953733087339
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       8.542616e+01  2.488688e+00  1.636521e+00  5.014068e-01   
Green      1.045830e-02  4.420662e+01  1.036242e+01  2.852778e-01   
Magenta    4.267682e-05  2.097224e+00  2.359948e+01  4.505772e-02   
Orange     8.649125e+00  3.473434e+01  4.706110e+01  8.617173e+01   
Chocolate  5.914212e+00  1.647313e+01  1.734048e+01  1.299652e+01   
Purple     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Deeppink   4.169916e-24  6.530265e-14  1.313910e-12  3.465883e-10   

                  Peak 5     Peak 6     Peak 7  
Blue        6.845434e-01   0.384477   0.221157  
Green       7.896772e-03   0.000123   0.000020  
Magenta     3.309033e-04   0.000006   

In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] +3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5197665347482896
0.5261623206171157
1.418435212942602
1.4268452027490879
1.742205617710434
1.8766697024198624
3.5718176470437135
3.5260331238438454
1.4522638914411639
1.4527454734126097
1.438471204234773
1.4379801030298183
2.1610281698452436
2.1125821321420415
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       8.446565e+01  2.591582e+00  1.697454e+00  5.529974e-01   
Green      1.172933e-02  4.002496e+01  1.192220e+01  3.375047e-01   
Magenta    4.732104e-05  3.721707e+00  1.870442e+01  6.207347e-02   
Orange     9.218164e+00  3.650896e+01  4.968790e+01  8.471053e+01   
Chocolate  6.304415e+00  1.715279e+01  1.798802e+01  1.433689e+01   
Purple     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Deeppink   6.514536e-24  9.789057e-14  1.956018e-12  5.447632e-10   

                 Peak 5     Peak 6     Peak 7  
Blue       6.928641e-01   0.383968   0.249813  
Green      8.438042e-03   0.000129   0.000024  
Magenta    3.569264e-04   0.000007   0.0

## SII

In [29]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/Fit/SII_fit.csv", index_col = 0)

In [30]:
fit

,Component 1,Component 2,Component 3,Component 4,Component 5,Component 6,Component 7
I_e,0.0120,0.0483,0.0655,0.0246,0.0762,1.2477,0.1425
r_e,10.9539,2.3054,0.6140,49.8577,10.5707,0.0473,4.3056
n,1.2062,0.5720,1.0272,1.9156,0.4962,0.6764,0.6417
x_0,19.0891,49.5398,53.4078,61.0920,76.8003,89.4672,95.9710


In [31]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.10990981906401957
0.10914938056817869
0.2547754339941149
0.2548150156197585
0.30735028037554724
0.33075118231531775
0.5986566071170907
0.5880332122495757
0.27319759465248034
0.2733157384253828
0.1828035784649943
0.2847651124759162
0.4001037441690612
0.3997122488028005
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       6.924990e+01  2.953563e-01  1.447093e-01  2.674591e-02   
Green      8.385591e-27  3.600978e+01  6.028032e+00  7.642574e-05   
Magenta    6.175907e-34  2.445966e-01  1.976229e+01  1.973826e-06   
Orange     3.075010e+01  6.284034e+01  7.230159e+01  9.392989e+01   
Chocolate  1.957913e-07  6.099191e-01  1.763374e+00  6.043282e+00   
Purple     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Deeppink   1.142159e-34  2.993901e-15  3.938599e-13  2.277865e-09   

                 Peak 5        Peak 6        Peak 7  
Blue       9.382531e-03  3.063201e-03  7.052194e-04  
Green      3.166583e-22  5.654428e-46  6.996921e-60  
Magenta    1.2

In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.3226933246086599
0.3206129007392483
0.7542489762431068
0.7543191925662536
0.9079816209242949
0.9659084651299503
1.7441638941685234
1.7153145720943792
0.8188420892518878
0.8191972669260927
0.5558627788641775
0.8137267855018258
1.1744347548532168
1.1734840209374426
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       6.856313e+01  3.006718e-01  1.475934e-01  2.765162e-02   
Green      1.128950e-25  3.458571e+01  6.733722e+00  1.535790e-04   
Magenta    2.354942e-33  6.600750e-01  1.760166e+01  7.373571e-06   
Orange     3.143687e+01  6.381954e+01  7.369462e+01  9.371424e+01   
Chocolate  2.269711e-07  6.340017e-01  1.822400e+00  6.257945e+00   
Purple     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Deeppink   2.342640e-34  4.570775e-15  5.797115e-13  3.146896e-09   

                 Peak 5        Peak 6        Peak 7  
Blue       9.425104e-03  3.032291e-03  7.231097e-04  
Green      2.525343e-21  1.628008e-44  3.758458e-58  
Magenta    4.081845

In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5104197357305248
0.507947658133139
1.2244732633245576
1.2267517242885657
1.473457954350009
1.5357545379595758
2.708175256806041
2.6741635901151204
1.3614799243909714
1.3620753230582054
0.9586091658116833
1.2455867209643012
2.151064977746879
2.151107802761823
                 Peak 1        Peak 2        Peak 3        Peak 4  \
Blue       6.682979e+01  3.124815e-01  1.533664e-01  3.000373e-02   
Green      2.962262e-24  3.104984e+01  8.448054e+00  5.532359e-04   
Magenta    1.776758e-32  2.042453e+00  1.300035e+01  6.606619e-05   
Orange     3.317021e+01  6.590095e+01  7.644045e+01  9.315229e+01   
Chocolate  3.195530e-07  6.942741e-01  1.957783e+00  6.817087e+00   
Purple     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Deeppink   8.205456e-34  1.032910e-14  1.232773e-12  6.154397e-09   

                 Peak 5        Peak 6        Peak 7  
Blue       9.539522e-03  2.956990e-03  7.571183e-04  
Green      4.582120e-20  1.101415e-42  4.357451e-56  
Magenta    3.051827e-21 

## OIII

In [24]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_1/FORS2.2022-01-07T06_54_16.725/Fit/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.1648847563844574
0.16775220171475325
0.3524280149688202
0.3541797559192451
0.38848462739771056
0.36391845782043164
0.7275200161565702
0.755030356684845
0.2953660561644149
0.29577221077178617
0.3892789327349589
0.38939050888436455
0.7918798350525588
0.9046178745177544
                 Peak 1     Peak 2     Peak 3     Peak 4         Peak 5  \
Blue       8.696014e+01   1.123683   0.808204   0.225619   1.995130e-01   
Green      1.352172e-10  52.207366  13.302206   0.003820   4.588716e-10   
Magenta    4.694988e-05   1.149379  31.199623   0.013167   1.797992e-04   
Orange     9.784420e+00  33.720088  41.675226  87.455406   2.809487e+01   
Chocolate  3.255390e+00  11.799248  13.014374  12.300906   7.160607e+01   
Purple     0.000000e+00   0.000000   0.000000   0.000000  1.662763e-191   
Deeppink   4.657923e-06   0.000237   0.000367   0.001082   9.936875e-02   

                 Peak 6        Peak 7  
Blue       6.783257e-02  2.412013e-02  
Green      1.968205e-17  5.065840e-21  
Magenta  

In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.4779252706857313
0.4848369218367543
1.0314938982625053
1.034238645323783
1.1324757201049889
1.0691306636592497
2.1085646808509853
2.176943326000286
0.8819453102319165
0.883002586142307
1.1570188364490461
1.1573862250441478
2.198329764258552
2.47812830892635
                 Peak 1     Peak 2     Peak 3     Peak 4         Peak 5  \
Blue       8.649144e+01   1.153877   0.833128   0.233850   2.006539e-01   
Green      1.940674e-10  50.015381  14.831943   0.004835   6.159377e-10   
Magenta    4.913000e-05   2.043984  27.832232   0.014559   1.835243e-04   
Orange     1.013507e+01  34.679944  43.095739  87.002049   2.829353e+01   
Chocolate  3.373435e+00  12.106568  13.406577  12.743575   7.140389e+01   
Purple     0.000000e+00   0.000000   0.000000   0.000000  1.173137e-102   
Deeppink   4.846222e-06   0.000245   0.000381   0.001132   1.017412e-01   

                 Peak 6        Peak 7  
Blue       6.852360e-02  2.608573e-02  
Green      2.796281e-17  7.852279e-21  
Magenta    7.550643

In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.7342364213582335
0.7414414212203154
1.6362482587915441
1.6298368971240642
1.816375937147644
1.737451971628203
3.249765768958804
3.3236487150982925
1.454022885148058
1.4553411659175954
1.879124851887739
1.8792279698404708
3.0510846417330133
3.3540547045755993
                 Peak 1     Peak 2     Peak 3     Peak 4        Peak 5  \
Blue       8.530978e+01   1.218285   0.869619   0.253795  2.033928e-01   
Green      4.072051e-10  44.387141  18.175181   0.008393  1.198379e-09   
Magenta    5.485025e-05   4.875734  21.628534   0.018873  1.936032e-04   
Orange     1.101866e+01  36.764194  45.358499  85.905242  2.878463e+01   
Chocolate  3.671504e+00  12.754383  13.967762  13.812438  7.090378e+01   
Purple     0.000000e+00   0.000000   0.000000   0.000000  1.173496e-51   
Deeppink   5.330222e-06   0.000263   0.000405   0.001259  1.080114e-01   

                 Peak 6        Peak 7  
Blue       7.047547e-02  3.138981e-02  
Green      6.097919e-17  2.074380e-20  
Magenta    7.936582e-06  1